<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/193-Langchain-RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Necessary Libraries
First, we need to install the required libraries: `langchain_community` for web loading and text splitting, `chromadb` for the vector store, and `langchain_openai` for embeddings and the language model. If you prefer to use open-source models, you can substitute `langchain_openai` with libraries like `langchain_huggingface` and `sentence-transformers`.



In [ ]:
!pip install -qqq langchain_community chromadb langchain-openai langchain

### 2. Set Up API Key (if using OpenAI)
If you plan to use OpenAI's models, you'll need to set up your API key. In Colab, you can store it securely in the secrets manager and access it as shown below. If you're using a different LLM provider or a local model, you'll configure that instead.

In [2]:
# Used to securely store your API key
from google.colab import userdata

import os

# Set your OpenAI API key. If running in Colab, store it securely in secrets.
# If running locally or programmatically, you will need to set it as an environment variable.
# For example, before running this cell, you might do:
# os.environ['OPENAI_API_KEY'] = 'your_openai_api_key_here'

if 'OPENAI_API_KEY' not in os.environ:
    # This line attempts to get the key from Colab secrets, which might time out
    # if not running interactively in the Colab UI.
    try:
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception as e:
        print(f"Warning: Could not retrieve API key from Colab secrets: {e}")
        print("Please ensure your OPENAI_API_KEY is set as an environment variable or in Colab secrets.")

if 'OPENAI_API_KEY' in os.environ:
    print("OPENAI_API_KEY is set.")
else:
    print("OPENAI_API_KEY is NOT set. Please set it to proceed.")

OPENAI_API_KEY is set.


### 3. Load the Document from the URL
We'll use `WebBaseLoader` from `langchain_community` to fetch the content from the specified URL.

In [3]:
from langchain_community.document_loaders import WebBaseLoader

# Define the URL to load
url = "http://www.paulgraham.com/fr.html"

# Initialize the WebBaseLoader with the URL
loader = WebBaseLoader(url)

# Load the documents
docs = loader.load()

print(f"Loaded {len(docs)} document(s).")
print("First 500 characters of the document:\n")
print(docs[0].page_content[:500])


Loaded 1 document(s).
First 500 characters of the document:

How to Raise Money



Want to start a startup?  Get funded by
Y Combinator.




September 2013Most startups that raise money do it more than once.  A typical
trajectory might be (1) to get started with a few tens of thousands
from something like Y Combinator or individual angels, then 
(2) raise a few hundred thousand to a few million to build the company,
and then (3) once the company is clearly succeeding, raise one or
more later rounds to accelerate growth.Reality can be messier.  Some compan


In [4]:
len(docs)

1

### 4. Split the Document into Chunks
To effectively retrieve relevant information, we need to split the large document into smaller, manageable chunks. We'll use `RecursiveCharacterTextSplitter` for this.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Maximum size of each chunk
    chunk_overlap=200,    # Overlap between chunks to maintain context
    add_start_index=True, # Add a metadata field for the start index of the chunk
)

# Split the documents
all_splits = text_splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks.")
print("First chunk:\n")
print(all_splits[0].page_content[:500])


Split into 80 chunks.
First chunk:

How to Raise Money



Want to start a startup?  Get funded by
Y Combinator.


### 5. Create Embeddings and a Vector Store
Next, we'll generate vector embeddings for each document chunk using OpenAIEmbeddings and store them in a Chroma vector database. This allows us to perform semantic search on the document content.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embeddings model
embeddings = OpenAIEmbeddings()

# Create a Chroma vector store from the document chunks and embeddings
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)

print("Vector store created successfully.")


Vector store created successfully.


In [7]:
print(embeddings.model)


text-embedding-ada-002


### 6. Set Up the RAG Chain
Now, we'll combine the retriever (which fetches relevant document chunks from the vector store) with a language model (OpenAI's ChatOpenAI) to build our RAG chain. This chain will retrieve relevant context and then use it to answer user questions.

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# Convert the vector store into a retriever
retriever = vectorstore.as_retriever()

In [9]:


# Define the RAG prompt template directly (bypassing hub.pull)
# This is a common RAG prompt pattern
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question based on the below context:\n\n{context}"),
    ("human", "{question}"),
])

# Define format_docs BEFORE it is used in rag_chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construct the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


### 7. Ask a Question
Finally, let's test our RAG system by asking a question related to the content of the Paul Graham essay.

In [11]:
question = "What are some key insights about founding a startup mentioned in the document?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)


Question: What are some key insights about founding a startup mentioned in the document?

Answer:
Some key insights about founding a startup mentioned in the document include:

1. **Focus on Fundraising**: Startups often face challenges related to fundraising, and it's crucial to secure commitments from investors as soon as possible. Delays can lead to changes in investor interest.

2. **Timely Collection of Funds**: Once an investor commits, founders should understand the timeline for receiving the funds and actively manage the process to ensure the money is transferred.

3. **Beware of Buyer’s Remorse**: Inexperienced investors are more likely to have second thoughts after committing. Founders should be cautious and prefer investors who take a leading role in funding.

4. **External Constraints**: Inexperienced founders should impose rules and constraints on themselves to navigate the fundraising process effectively, as trusting one's intuition may lead to poor decisions.

5. **Inves

In [10]:
question = "저자의 이름은?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 저자의 이름은?

Answer:
저자의 이름은 명시되어 있지 않습니다. 그러나 문맥에서 여러 사람들의 이름이 언급되고 있으며, 이들은 저자의 원고를 읽어본 사람들입니다.


In [12]:
question = "문자의 핵심 메세지를 100자로 정리해 주세요."
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 문자의 핵심 메세지를 100자로 정리해 주세요.

Answer:
투자자 미팅은 너무 빡빡하게 잡지 말고, 피치를 발전시킬 시간을 가져야 한다. 투자자의 관심은 가짜일 수 있으므로, 확실한 약속이 있을 때까지 최악의 상황을 가정하라.


In [13]:
question = "고양이 다리의 갯수는?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 고양이 다리의 갯수는?

Answer:
고양이는 일반적으로 4개의 다리를 가지고 있습니다.
